<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/%EA%B9%80%EC%98%81%EB%B9%88_cnn_%EC%8B%A0%ED%98%B8%EB%93%B1%EC%9D%B8%EC%8B%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from PIL import Image
import io
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import os

def detect_traffic_light_cnn(image, model=None, min_area=100, max_area=8000, canny_low=50, canny_high=150, circularity_threshold=0.5, use_color_filter=True, use_cnn=True):
    """
    CNN 기능이 추가된 신호등 검출 함수
    기존 코드 + CNN 분류 결합
    """

    # 색상 필터링
    if use_color_filter:
        print("Step 0: 신호등 색상 필터링 시작...")
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        # 색상 범위 정의
        red_lower1 = np.array([0, 50, 50])
        red_upper1 = np.array([15, 255, 255])
        red_lower2 = np.array([170, 50, 50])
        red_upper2 = np.array([180, 255, 255])
        red_lower3 = np.array([10, 70, 50])
        red_upper3 = np.array([20, 255, 255])

        yellow_lower = np.array([20, 80, 80])
        yellow_upper = np.array([35, 255, 255])

        green_lower = np.array([35, 50, 50])
        green_upper = np.array([90, 255, 255])

        blue_lower = np.array([85, 50, 50])
        blue_upper = np.array([100, 255, 255])

        # 마스크 생성
        red_mask1 = cv2.inRange(hsv, red_lower1, red_upper1)
        red_mask2 = cv2.inRange(hsv, red_lower2, red_upper2)
        red_mask3 = cv2.inRange(hsv, red_lower3, red_upper3)
        red_mask = cv2.bitwise_or(cv2.bitwise_or(red_mask1, red_mask2), red_mask3)

        yellow_mask = cv2.inRange(hsv, yellow_lower, yellow_upper)
        green_mask = cv2.inRange(hsv, green_lower, green_upper)
        blue_mask = cv2.inRange(hsv, blue_lower, blue_upper)

        traffic_light_mask = red_mask
        traffic_light_mask = cv2.bitwise_or(traffic_light_mask, yellow_mask)
        traffic_light_mask = cv2.bitwise_or(traffic_light_mask, green_mask)
        traffic_light_mask = cv2.bitwise_or(traffic_light_mask, blue_mask)

        color_filtered = cv2.bitwise_and(image, image, mask=traffic_light_mask)
        gray = cv2.cvtColor(color_filtered, cv2.COLOR_BGR2GRAY)
        print("Step 0: 색상 필터링 완료")
    else:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        print("Step 1: 컬러 → 흑백 변환 완료")

    # 노이즈 제거 및 엣지 검출
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    print("Step 2: 노이즈 제거 완료")

    edges = cv2.Canny(blurred, canny_low, canny_high)
    print(f"Step 3: 엣지 검출 완료 ({canny_low}-{canny_high})")

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print(f"Step 4: {len(contours)}개 윤곽선 발견")

    # CNN을 이용한 후보 검증
    traffic_lights = []
    image_height = image.shape[0]

    for i, contour in enumerate(contours):
        # 기존 필터링 조건들
        area = cv2.contourArea(contour)
        if area < min_area or area > max_area:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        center_y = y + h // 2
        if center_y < image_height * 0.15 or center_y > image_height * 0.75:
            continue

        aspect_ratio = float(w) / h
        if aspect_ratio > 0.9:
            continue

        perimeter = cv2.arcLength(contour, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)
        if circularity < circularity_threshold:
            continue

        # CNN 검증 단계
        if use_cnn and model is not None:
            roi = image[y:y+h, x:x+w]
            if roi.size > 0:
                roi_resized = cv2.resize(roi, (32, 32))
                roi_normalized = roi_resized.astype('float32') / 255.0
                roi_input = np.expand_dims(roi_normalized, axis=0)

                prediction = model.predict(roi_input, verbose=0)
                confidence = prediction[0][1]

                print(f"CNN 검증 - 후보 {i+1}: 신호등 확률 = {confidence:.3f}")

                if confidence > 0.7:
                    traffic_lights.append((x, y, w, h, confidence))
                    print(f"  ✅ 신호등으로 인정!")
                else:
                    print(f"  ❌ 신호등이 아님 (낮은 확률)")
        else:
            traffic_lights.append((x, y, w, h, 1.0))

    print(f"최종 결과: {len(traffic_lights)}개 신호등 발견")
    return traffic_lights, edges

def create_cnn_model():
    """
    신호등 분류를 위한 CNN 모델 생성
    입력: 32x32x3 컬러 이미지
    출력: [not_traffic_light, traffic_light] 확률
    """
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(2, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print("🧠 CNN 모델 생성 완료!")
    model.summary()
    return model

def create_real_dataset_from_detections(image, detections):
    """
    실제 검출된 후보들로부터 데이터셋 생성
    검출된 것 = 신호등(1), 주변 배경 = 비신호등(0)
    """
    print("📦 실제 이미지 기반 데이터셋 생성 중...")

    X = []
    y = []

    # 검출된 후보들 → 신호등 클래스 (1)
    for detection in detections:
        if len(detection) >= 4:
            x, y_pos, w, h = detection[:4]
            roi = image[y_pos:y_pos+h, x:x+w]
            if roi.size > 0:
                roi_resized = cv2.resize(roi, (32, 32))
                roi_normalized = roi_resized.astype('float32') / 255.0
                X.append(roi_normalized)
                y.append(1)

    # 랜덤 배경 영역들 → 비신호등 클래스 (0)
    height, width = image.shape[:2]
    background_samples = len(detections) * 3

    for _ in range(background_samples):
        x = np.random.randint(0, max(1, width - 32))
        y_pos = np.random.randint(0, max(1, height - 32))

        # 검출된 신호등 영역과 겹치지 않는지 확인
        is_overlap = False
        for detection in detections:
            if len(detection) >= 4:
                det_x, det_y, det_w, det_h = detection[:4]
                if (x < det_x + det_w and x + 32 > det_x and
                    y_pos < det_y + det_h and y_pos + 32 > det_y):
                    is_overlap = True
                    break

        if not is_overlap:
            roi = image[y_pos:y_pos+32, x:x+32]
            if roi.shape == (32, 32, 3):
                roi_normalized = roi.astype('float32') / 255.0
                X.append(roi_normalized)
                y.append(0)

    X = np.array(X)
    y = np.array(y)

    print(f"📊 실제 데이터셋 생성 완료: {len(X)}개 샘플 (신호등:{np.sum(y==1)}, 배경:{np.sum(y==0)})")
    return X, y

def train_cnn_with_real_data(image, initial_detections):
    """
    실제 이미지에서 검출된 결과를 바탕으로 CNN 모델 훈련
    """
    print("🎓 실제 데이터 기반 CNN 모델 훈련 시작!")

    model = create_cnn_model()
    X, y = create_real_dataset_from_detections(image, initial_detections)

    if len(X) < 10:
        print("⚠️ 데이터가 부족합니다. 기본 가중치를 사용합니다.")
        return model

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("🔄 빠른 훈련 시작 (5 epochs)...")
    history = model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=16,
        validation_data=(X_test, y_test) if len(X_test) > 0 else None,
        verbose=1
    )

    if len(X_test) > 0:
        test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
        print(f"🎯 테스트 정확도: {test_acc:.3f}")

    return model

def show_training_samples(image, detections):
    """
    CNN 훈련에 사용되는 실제 샘플들을 보여주는 함수
    """
    print("🔍 CNN 훈련 샘플 미리보기...")

    if len(detections) == 0:
        print("⚠️ 표시할 샘플이 없습니다.")
        return

    # 신호등 샘플들 추출
    traffic_samples = []
    for detection in detections[:6]:
        if len(detection) >= 4:
            x, y, w, h = detection[:4]
            roi = image[y:y+h, x:x+w]
            if roi.size > 0:
                roi_resized = cv2.resize(roi, (32, 32))
                traffic_samples.append(roi_resized)

    # 배경 샘플들 추출
    height, width = image.shape[:2]
    background_samples = []
    for _ in range(6):
        x = np.random.randint(0, max(1, width - 32))
        y_pos = np.random.randint(0, max(1, height - 32))
        roi = image[y_pos:y_pos+32, x:x+32]
        if roi.shape == (32, 32, 3):
            background_samples.append(roi)

    # 시각화
    plt.figure(figsize=(15, 6))

    for i, sample in enumerate(traffic_samples):
        plt.subplot(2, 6, i + 1)
        plt.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
        plt.title(f'신호등 {i+1}', fontsize=10)
        plt.axis('off')

    for i, sample in enumerate(background_samples):
        plt.subplot(2, 6, 6 + i + 1)
        plt.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
        plt.title(f'배경 {i+1}', fontsize=10)
        plt.axis('off')

    plt.suptitle('CNN 훈련 샘플: 위쪽=신호등(라벨=1), 아래쪽=배경(라벨=0)', fontsize=14)
    plt.tight_layout()
    plt.show()

def draw_detections(image, detections, show_confidence=True):
    """
    검출 결과를 이미지에 그리는 함수 (CNN 신뢰도 표시 포함)
    """
    result = image.copy()
    height, width = image.shape[:2]
    step = height // 10

    # 격자 그리기
    for i in range(1, 11):
        y = i * step
        cv2.line(result, (0, y), (width, y), (255, 0, 0), 1)
        cv2.putText(result, f'y={y}', (5, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

    # 초록 반투명 박스
    overlay = result.copy()
    alpha = 0.3
    cv2.rectangle(overlay, (0, 90), (width, 120), (0, 255, 0), -1)
    cv2.rectangle(overlay, (0, 260), (width, 420), (0, 255, 0), -1)
    cv2.addWeighted(overlay, alpha, result, 1 - alpha, 0, result)

    # 신호등 박스 그리기
    for i, detection in enumerate(detections):
        if len(detection) == 5:
            x, y, w, h, confidence = detection
            confidence_text = f'{confidence:.2f}'
        else:
            x, y, w, h = detection[:4]
            confidence_text = '1.00'

        center_y = y + h // 2
        if (90 <= center_y <= 120) or (260 <= center_y <= 420):
            cv2.rectangle(result, (x, y), (x + w, y + h), (0, 255, 0), 2)

            if show_confidence:
                cv2.putText(result, f'TL{i+1} ({confidence_text})', (x, y - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            else:
                cv2.putText(result, f'Traffic Light {i+1}', (x, y - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    return result

def upload_and_detect_cnn():
    """
    실제로 동작하는 CNN 통합 신호등 검출 함수
    1단계: 기존 방식으로 초기 검출
    2단계: 검출 결과로 CNN 훈련 후 재검출
    """
    print("🚦🧠 실제 CNN 통합 신호등 인식 시작!")

    print("1️⃣ 이미지 업로드...")
    uploaded = files.upload()

    for filename in uploaded.keys():
        print(f"\n🖼️ 처리중: {filename}")

        # 이미지 로드
        image_data = uploaded[filename]
        image = Image.open(io.BytesIO(image_data))
        image_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

        # 크기 조정
        height, width = image_cv.shape[:2]
        if width > 1200:
            ratio = 1200 / width
            image_cv = cv2.resize(image_cv, (1200, int(height * ratio)))

        # 1단계: 기존 방식으로 초기 검출
        print("2️⃣ 1단계: 기존 방식으로 초기 검출...")
        initial_detections, edges = detect_traffic_light_cnn(
            image_cv,
            model=None,
            min_area=125,
            max_area=8000,
            canny_low=30,
            canny_high=180,
            circularity_threshold=0.15,
            use_color_filter=True,
            use_cnn=False
        )

        print(f"   📊 1단계 결과: {len(initial_detections)}개 후보 발견")

        # 2단계: 실제 데이터로 CNN 훈련
        if len(initial_detections) > 0:
            print("3️⃣ 2단계: CNN 훈련 샘플 확인...")
            show_training_samples(image_cv, initial_detections)

            print("4️⃣ 3단계: 실제 데이터로 CNN 훈련...")
            model = train_cnn_with_real_data(image_cv, initial_detections)

            # 3단계: CNN으로 재검출
            print("5️⃣ 4단계: CNN으로 재검출...")
            final_detections, _ = detect_traffic_light_cnn(
                image_cv,
                model=model,
                min_area=80,
                max_area=10000,
                canny_low=30,
                canny_high=180,
                circularity_threshold=0.1,
                use_color_filter=True,
                use_cnn=True
            )

            print(f"   🎯 최종 결과: {len(final_detections)}개 신호등 검출")
        else:
            print("⚠️ 초기 검출 결과가 없어 CNN 훈련을 건너뜁니다.")
            final_detections = initial_detections
            model = None

        # 결과 시각화
        result_initial = draw_detections(image_cv, initial_detections, show_confidence=False)
        result_final = draw_detections(image_cv, final_detections, show_confidence=True)

        plt.figure(figsize=(20, 10))

        plt.subplot(2, 2, 1)
        plt.imshow(cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB))
        plt.title(f'원본 이미지: {filename}')
        plt.axis('off')

        plt.subplot(2, 2, 2)
        plt.imshow(edges, cmap='gray')
        plt.title('Canny 엣지 검출')
        plt.axis('off')

        plt.subplot(2, 2, 3)
        plt.imshow(cv2.cvtColor(result_initial, cv2.COLOR_BGR2RGB))
        plt.title(f'1단계 (기존 방식): {len(initial_detections)}개')
        plt.axis('off')

        plt.subplot(2, 2, 4)
        plt.imshow(cv2.cvtColor(result_final, cv2.COLOR_BGR2RGB))
        plt.title(f'최종 (CNN 검증): {len(final_detections)}개')
        plt.axis('off')

        plt.tight_layout()
        plt.show()

        # 상세 결과 출력
        print("\n📋 상세 검출 결과:")
        print("=" * 50)
        print(f"🔍 1단계 (기존 방식): {len(initial_detections)}개 후보")
        for i, detection in enumerate(initial_detections):
            if len(detection) >= 4:
                x, y, w, h = detection[:4]
                print(f"   후보 {i+1}: 위치=({x}, {y}), 크기=({w}x{h})")

        print(f"\n🧠 최종 (CNN 검증): {len(final_detections)}개 신호등")
        for i, detection in enumerate(final_detections):
            if len(detection) == 5:
                x, y, w, h, confidence = detection
                print(f"   🚦 신호등 {i+1}: 위치=({x}, {y}), 크기=({w}x{h}), 신뢰도={confidence:.3f}")
            else:
                x, y, w, h = detection[:4]
                print(f"   🚦 신호등 {i+1}: 위치=({x}, {y}), 크기=({w}x{h})")

        # 검출 품질 평가
        if model is not None and len(initial_detections) > 0:
            retention_rate = len(final_detections) / len(initial_detections) * 100
            print(f"\n🎯 검출 품질 평가:")
            print(f"   - 초기 후보: {len(initial_detections)}개")
            print(f"   - CNN 필터링 후: {len(final_detections)}개")
            print(f"   - 유지율: {retention_rate:.1f}%")

            if retention_rate > 80:
                print("   ✅ 높은 신뢰도 (대부분 신호등으로 판정)")
            elif retention_rate > 50:
                print("   🔶 중간 신뢰도 (일부 노이즈 제거)")
            else:
                print("   🔴 낮은 신뢰도 (많은 후보가 노이즈로 판정)")

        print("=" * 50)

# 실행 옵션
print("🚦🧠 CNN 통합 신호등 인식 프로그램")
print("1️⃣ upload_and_detect_cnn() - CNN 통합 검출 (추천!)")
print()
print("💡 추천: upload_and_detect_cnn() 를 실행하세요!")

# CNN 통합 버전 바로 실행
upload_and_detect_cnn()